In [2]:
## First mount your drive to get the datasets.
import pandas as pd
import numpy as np
import sklearn
from sklearn.model_selection import train_test_split
from matplotlib import colormaps
import seaborn as sns
import matplotlib.pyplot as plt
from statsmodels.stats.outliers_influence import variance_inflation_factor
from statsmodels.tools.tools import add_constant
from sklearn.base import BaseEstimator, TransformerMixin
pd.set_option('display.max_columns',None)

In [22]:
csc_raw_train = pd.read_csv('/content/drive/MyDrive/credit_score_project/datasets/Credit_Score_Classification/train.csv')
# csc_raw_test = pd.read_csv('/content/drive/MyDrive/credit_score_project/datasets/Credit_Score_Classification/test.csv')
df = csc_raw_train.copy()
# df_test = csc_raw_test.copy()
# df = pd.concat([df_train,df_test],axis=0)

/tmp/ipython-input-2292205840.py:1: DtypeWarning: Columns (26) have mixed types. Specify dtype option on import or set low_memory=False.
  csc_raw_train = pd.read_csv('/content/drive/MyDrive/credit_score_project/datasets/Credit_Score_Classification/train.csv')


In [23]:
class drop_columns(BaseEstimator, TransformerMixin):

  def __init__(self,columns_to_drop):
    self.columns_to_drop = columns_to_drop

  def fit(self, X, y=None):
    return self

  def transform(self,X):
    return X.drop(columns=self.columns_to_drop)

col_to_drop = ['ID','Name','SSN']
dropper = drop_columns(col_to_drop)
df = dropper.transform(df)
df_copy1 = df.copy()
df_copy1.shape

(100000, 25)

Below cell contains cleaning taken from Kaggle notebook.

In [24]:
def text_cleaning(data):
    if data is np.nan or not isinstance(data, str):
        return data
    else:
        return str(data).strip('_ ,"')
df = df_copy1.applymap(text_cleaning).replace(['', 'nan', '!@9#%8', '#F%$D@*&8'], np.nan)

df['Customer_ID']             = df.Customer_ID.apply(lambda x: int(x[4:], 16))
df['Month']                   = pd.to_datetime(df.Month, format='%B').dt.month
df['Age']                     = df.Age.astype(int)
df['Annual_Income']           = df.Annual_Income.astype(float)
df['Num_of_Loan']             = df.Num_of_Loan.astype(int)
df['Num_of_Delayed_Payment']  = df.Num_of_Delayed_Payment.astype(float)
df['Changed_Credit_Limit']    = df.Changed_Credit_Limit.astype(float)
df['Outstanding_Debt']        = df.Outstanding_Debt.astype(float)
df['Amount_invested_monthly'] = df.Amount_invested_monthly.astype(float)
df['Monthly_Balance']         = df.Monthly_Balance.astype(float)
def Month_Converter(x):
    if pd.notnull(x):
        num1 = int(x.split(' ')[0])
        num2 = int(x.split(' ')[3])

        return (num1*12)+num2
    else:
        return x

# Month_Converter('3 Years and 1 Months')
df['Credit_History_Age'] = df.Credit_History_Age.apply(lambda x: Month_Converter(x)).astype(float)
df['Payment_of_Min_Amount'] = df['Payment_of_Min_Amount'].replace('NM','No')
df['Type_of_Loan'] = df['Type_of_Loan'].apply(lambda x: x.lower().replace('and ', '').replace(', ', ',').strip() if pd.notna(x) else x)
df.loc[df['Type_of_Loan']=='not specified','Type_of_Loan'] = np.nan

/tmp/ipython-input-4087233130.py:6: FutureWarning: DataFrame.applymap has been deprecated. Use DataFrame.map instead.
  df = df_copy1.applymap(text_cleaning).replace(['', 'nan', '!@9#%8', '#F%$D@*&8'], np.nan)


In [27]:
# Reassign and Show Function
def Object_NaN_Values_Reassign_Group_Mode(df, groupby, column, inplace=True):
    import scipy.stats as stats
    # Actual replce function
    def make_NaN_and_fill_mode(df, groupby, column, inplace=True):

        if df[column].isin([None]).sum():
            df[column][df[column].isin([None])] = np.nan


        result = df.groupby(groupby)[column].transform(lambda x: x.fillna(x.mode()[0] if len(x.mode())!=0 else np.nan))


        if inplace:
            df[column]=result
        else:
            return result

    # This part is Just for the print purposes.
    if inplace:

        if df[column].value_counts(dropna=False).index.isna().sum():
            x = df[column].value_counts(dropna=False).loc[[np.nan]]
            print(f'\nBefore Assigning: {column}:', f'have {x.values[0]} NaN Values', end='\n')

        a = df.groupby(groupby)[column].apply(list)
        print(f'\nBefore Assigning Example {column}:\n', a.head().values, sep='\n', end='\n')


        make_NaN_and_fill_mode(df, groupby, column, inplace)


        if df[column].value_counts(dropna=False).index.isna().sum():
            y = df[column].value_counts(dropna=False).loc[[np.nan]]
            print(f'\nAfter Assigning: {column}:', f'have {y.values[0]} NaN Values', end='\n')

        b = df.groupby(groupby)[column].apply(list)
        print(f'\nAfter Assigning Example {column}:\n', b.head().values, sep='\n', end='\n')
    else:

        return make_NaN_and_fill_mode(df, groupby, column, inplace)

Object_NaN_Values_Reassign_Group_Mode(df, 'Customer_ID', 'Occupation')
Object_NaN_Values_Reassign_Group_Mode(df, 'Customer_ID', 'Credit_Mix')
Object_NaN_Values_Reassign_Group_Mode(df, 'Customer_ID', 'Payment_Behaviour')
Object_NaN_Values_Reassign_Group_Mode(df, 'Customer_ID', 'Type_of_Loan') # fill type of loan values if out of 8 only some are none

df['Credit_History_Age'] = df.groupby('Customer_ID')['Credit_History_Age'].transform(lambda x: x.interpolate().bfill().ffill())

df['spent_behaviour'] = df['Payment_Behaviour'].str.replace('_value_payments','').str.split('_spent_').apply(lambda x: x[0])
df['spent_value'] = df['Payment_Behaviour'].str.replace('_value_payments','').str.split('_spent_').apply(lambda x: x[1])


Before Assigning: Occupation: have 7062 NaN Values

Before Assigning Example Occupation:

[list(['Journalist', 'Journalist', 'Journalist', 'Journalist', 'Journalist', nan, 'Journalist', 'Journalist'])
 list(['Manager', 'Manager', nan, 'Manager', 'Manager', 'Manager', 'Manager', 'Manager'])
 list(['Developer', 'Developer', 'Developer', 'Developer', 'Developer', 'Developer', 'Developer', 'Developer'])
 list(['Accountant', nan, 'Accountant', 'Accountant', 'Accountant', 'Accountant', 'Accountant', 'Accountant'])
 list(['Writer', 'Writer', 'Writer', 'Writer', nan, 'Writer', 'Writer', 'Writer'])]

After Assigning Example Occupation:

[list(['Journalist', 'Journalist', 'Journalist', 'Journalist', 'Journalist', 'Journalist', 'Journalist', 'Journalist'])
 list(['Manager', 'Manager', 'Manager', 'Manager', 'Manager', 'Manager', 'Manager', 'Manager'])
 list(['Developer', 'Developer', 'Developer', 'Developer', 'Developer', 'Developer', 'Developer', 'Developer'])
 list(['Accountant', 'Accountant', 

/tmp/ipython-input-3809460993.py:11: FutureWarning: Downcasting object dtype arrays on .fillna, .ffill, .bfill is deprecated and will change in a future version. Call result.infer_objects(copy=False) instead. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  result = df.groupby(groupby)[column].transform(lambda x: x.fillna(x.mode()[0] if len(x.mode())!=0 else np.nan))



After Assigning: Type_of_Loan: have 12816 NaN Values

After Assigning Example Type_of_Loan:

[list(['credit-builder loan,payday loan', 'credit-builder loan,payday loan', 'credit-builder loan,payday loan', 'credit-builder loan,payday loan', 'credit-builder loan,payday loan', 'credit-builder loan,payday loan', 'credit-builder loan,payday loan', 'credit-builder loan,payday loan'])
 list(['home equity loan,mortgage loan,student loan', 'home equity loan,mortgage loan,student loan', 'home equity loan,mortgage loan,student loan', 'home equity loan,mortgage loan,student loan', 'home equity loan,mortgage loan,student loan', 'home equity loan,mortgage loan,student loan', 'home equity loan,mortgage loan,student loan', 'home equity loan,mortgage loan,student loan'])
 list([nan, nan, nan, nan, nan, nan, nan, nan])
 list(['credit-builder loan,student loan,not specified,student loan', 'credit-builder loan,student loan,not specified,student loan', 'credit-builder loan,student loan,not specified,stude

In [11]:
df.isnull().sum()

,0
Customer_ID,0
Month,0
Age,0
Occupation,0
Annual_Income,0
Monthly_Inhand_Salary,15002
Num_Bank_Accounts,0
Num_Credit_Card,0
Interest_Rate,0
Num_of_Loan,0


In [23]:
df.head()

,Customer_ID,Month,Age,Occupation,Annual_Income,Monthly_Inhand_Salary,Num_Bank_Accounts,Num_Credit_Card,Interest_Rate,Num_of_Loan,Type_of_Loan,Delay_from_due_date,Num_of_Delayed_Payment,Changed_Credit_Limit,Num_Credit_Inquiries,Credit_Mix,Outstanding_Debt,Credit_Utilization_Ratio,Credit_History_Age,Payment_of_Min_Amount,Total_EMI_per_month,Amount_invested_monthly,Payment_Behaviour,Monthly_Balance,Credit_Score,spent_behaviour,spent_value
0,3392,1,23,Scientist,19114.12,1824.843333,3,4,3,4,"[auto loan, credit-builder loan, personal loan...",3,7.0,11.27,4.0,Good,809.98,26.822620,265.0,No,49.574949,80.415295,High_spent_Small_value_payments,312.494089,Good,High,Small
1,3392,2,23,Scientist,19114.12,NaN,3,4,3,4,"[auto loan, credit-builder loan, personal loan...",-1,NaN,11.27,4.0,Good,809.98,31.944960,266.0,No,49.574949,118.280222,Low_spent_Large_value_payments,284.629162,Good,Low,Large
2,3392,3,-500,Scientist,19114.12,NaN,3,4,3,4,"[auto loan, credit-builder loan, personal loan...",3,7.0,NaN,4.0,Good,809.98,28.609352,267.0,No,49.574949,81.699521,Low_spent_Medium_value_payments,331.209863,Good,Low,Medium
3,3392,4,23,Scientist,19114.12,NaN,3,4,3,4,"[auto loan, credit-builder loan, personal loan...",5,4.0,6.27,4.0,Good,809.98,31.377862,268.0,No,49.574949,199.458074,Low_spent_Small_value_payments,223.451310,Good,Low,Small
4,3392,5,23,Scientist,19114.12,1824.843333,3,4,3,4,"[auto loan, credit-builder loan, personal loan...",6,NaN,11.27,4.0,Good,809.98,24.797347,269.0,No,49.574949,41.420153,High_spent_Medium_value_payments,341.489231,Good,High,Medium


In [7]:
num_cols = ['Month','Age','Annual_Income','Monthly_Inhand_Salary','Num_Bank_Accounts','Num_Credit_Card',
            'Interest_Rate','Num_of_Loan','Delay_from_due_date','Num_of_Delayed_Payment','Changed_Credit_Limit',
            'Num_Credit_Inquiries','Outstanding_Debt','Credit_Utilization_Ratio','Credit_History_Age',
            'Total_EMI_per_month','Amount_invested_monthly','Monthly_Balance']
total_negative_rows = (df[num_cols] < 0).any(axis=1).sum()
total_negative_rows

# If we consider this -ve values as nan then we should impute them. But question is
# whether it should be part of data cleaning or preprocessing. From what I can see is
# that if for a perticular record if I am to fill that value with same value that is
# present or given for the same customer ID that is not imputation it is just a cleaning.
# Based on this I am going ahead.

# There are total 7384 different rows of it. We are going to clean the numerical
# cols same as we have done for categorcal ones. But what if the -ve value appears
# in the testing data? or what if the input to a ML model is -ve value? how that
# scenario is going to be handled

np.int64(7384)

In [28]:
def fix_numeric_by_group(df, groupby, column):

    print(column,df.loc[df[column]<0, column].shape)

    df.loc[df[column]<0, column] = np.nan

    df[column] = df.groupby(groupby)[column]\
                    .transform(lambda x: x.fillna(x.median()))

    return df

for col in num_cols:
  fix_numeric_by_group(df,'Customer_ID',col)

Month (0,)
Age (886,)
Annual_Income (0,)
Monthly_Inhand_Salary (0,)
Num_Bank_Accounts (21,)
Num_Credit_Card (0,)
Interest_Rate (0,)
Num_of_Loan (3876,)
Delay_from_due_date (591,)
Num_of_Delayed_Payment (644,)
Changed_Credit_Limit (1586,)
Num_Credit_Inquiries (0,)
Outstanding_Debt (0,)
Credit_Utilization_Ratio (0,)
Credit_History_Age (0,)
Total_EMI_per_month (0,)
Amount_invested_monthly (0,)
Monthly_Balance (9,)


In [ ]:
df.isnull().sum()

In [ ]:
for col in num_cols:
  print(col,df[col].max())

In [ ]:
for col in num_cols:
  gdf = df[['Customer_ID',col]].copy()
  gdf['q1'] = gdf.groupby('Customer_ID')[col].transform(lambda x: x.quantile(0.25))
  gdf['q3'] = gdf.groupby('Customer_ID')[col].transform(lambda x: x.quantile(0.75))
  gdf['IQR'] = gdf['q3'] - gdf['q1']
  gdf[f'{col}_maxiqr'] = gdf['q3'] + 1.5 * gdf['IQR']
  gdf[f'{col}_miniqr'] = gdf['q1'] - 1.5 * gdf['IQR']

  y = gdf[(gdf[col]>gdf[f'{col}_maxiqr']) | (gdf[col]< gdf[f'{col}_miniqr'])]
  print (col,y.shape[0])

In [ ]:
# threshold = 40
# col = 'Num_Credit_Inquiries'
# exdf = df[['Customer_ID',col]].copy()

# exdf['std'] = exdf.groupby('Customer_ID')[col].transform(lambda x: x.std())
# exdf['median'] = exdf.groupby('Customer_ID')[col].transform(lambda x: x.median())
# exdf['value'] = (exdf[col] - exdf['median']).abs()
# exdf['mask'] = (exdf[col] - exdf['median']).abs() > threshold
# exdf[exdf['mask']==True].sort_values(by='value',ascending=True)

In [29]:

exdf = df.copy()
thresholds = {'Age':1,'Annual_Income':1,'Num_Bank_Accounts':20,'Num_Credit_Card':30,'Interest_Rate':35
              ,'Num_of_Loan':30,'Num_of_Delayed_Payment':10,'Num_Credit_Inquiries':40}

def fix_outliers_with_previous_value(df, groupby, column, threshold):

    def clean_group(x):

        # compute group statistics
        median = x.median()

        # detect outlier values
        mask = (x - median).abs() > threshold

        # Temporarily mark outliers as NaN
        x_clean = x.mask(mask)

        # Fill using previous valid value, fallback to next
        x_clean = x_clean.ffill().bfill()

        return x_clean

    df[column] = df.groupby(groupby)[column].transform(clean_group)

    return df
for col, threshold in thresholds.items():
    exdf = fix_outliers_with_previous_value(exdf,'Customer_ID',col,threshold)

In [ ]:
exdf.head(50)

In [ ]:
exdf.isnull().sum()

# Customer_ID	0
# Month	0
# Age	261
# Occupation	0
# Annual_Income	140
# Monthly_Inhand_Salary	0
# Num_Bank_Accounts	162
# Num_Credit_Card	274
# Interest_Rate	239
# Num_of_Loan	69
# Type_of_Loan	11408
# Delay_from_due_date	0
# Num_of_Delayed_Payment	90
# Changed_Credit_Limit	0
# Num_Credit_Inquiries	187
# Credit_Mix	0
# Outstanding_Debt	0
# Credit_Utilization_Ratio	0
# Credit_History_Age	0
# Payment_of_Min_Amount	0
# Total_EMI_per_month	0
# Amount_invested_monthly	0
# Payment_Behaviour	0
# Monthly_Balance	0
# Credit_Score	0


In [ ]:
for col,threshold in thresholds.items():
  gdf = exdf[['Customer_ID',col]].copy()
  gdf['q1'] = gdf.groupby('Customer_ID')[col].transform(lambda x: x.quantile(0.25))
  gdf['q3'] = gdf.groupby('Customer_ID')[col].transform(lambda x: x.quantile(0.75))
  gdf['IQR'] = gdf['q3'] - gdf['q1']
  gdf[f'{col}_maxiqr'] = gdf['q3'] + 1.5 * gdf['IQR']
  gdf[f'{col}_miniqr'] = gdf['q1'] - 1.5 * gdf['IQR']

  y = gdf[(gdf[col]>gdf[f'{col}_maxiqr']) | (gdf[col]< gdf[f'{col}_miniqr'])]
  print (col,y.shape[0])

  # Prev Values
# Month 0
# Age 7939
# Annual_Income 993
# Monthly_Inhand_Salary 775
# Num_Bank_Accounts 1560
# Num_Credit_Card 2507
# Interest_Rate 2022
# Num_of_Loan 472
# Delay_from_due_date 18909
# Num_of_Delayed_Payment 19445
# Changed_Credit_Limit 14599
# Num_Credit_Inquiries 7740
# Outstanding_Debt 0
# Credit_Utilization_Ratio 2151
# Credit_History_Age 0
# Total_EMI_per_month 3744
# Amount_invested_monthly 8234
# Monthly_Balance 4445

In [30]:
df_train_cleaned = exdf.drop(columns=['Payment_Behaviour']).copy()


In [ ]:
for col in num_cols:
  print(col,df_train_cleaned[col].max())

# Month 8.0
# Age 8698.0
# Annual_Income 24198062.0
# Monthly_Inhand_Salary 15204.633333333331
# Num_Bank_Accounts 1798.0
# Num_Credit_Card 1499.0
# Interest_Rate 5797.0
# Num_of_Loan 1496.0
# Delay_from_due_date 67.0
# Num_of_Delayed_Payment 4397.0
# Changed_Credit_Limit 36.97
# Num_Credit_Inquiries 2597.0
# Outstanding_Debt 4998.07
# Credit_Utilization_Ratio 50.00000000000001
# Credit_History_Age 404.0
# Total_EMI_per_month 82331.0
# Amount_invested_monthly 10000.0
# Monthly_Balance 1602.0405189622518

In [ ]:
df.describe()

In [32]:
df_train_cleaned.isnull().sum()

,0
Customer_ID,0
Month,0
Age,0
Occupation,0
Annual_Income,0
Monthly_Inhand_Salary,0
Num_Bank_Accounts,0
Num_Credit_Card,0
Interest_Rate,0
Num_of_Loan,0


In [31]:
df_train_cleaned.to_csv('/content/drive/MyDrive/credit_score_project/datasets_cleaned/csc_cleaned.csv')